In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *
from scipy.ndimage import gaussian_filter

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [3]:
#files_fdt = sorted(glob.glob('/home/ulyanov/data/cross/2024-11/fdt/*'))
files_fdt = sorted(glob.glob('/home/ulyanov/data/cross/*fdt*'))
files_fdt

['/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20250313T060009_V202602220258_0543130501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-stokes_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz']

In [4]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/cross/*hmi*'))
files_hmi

['/home/ulyanov/data/cross/hmi.M_45s.20240319_152445_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_45s.20241120_001630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_45s.20250313_060430_TAI.2.magnetogram.fits']

In [16]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xu, yu = s['xu'], s['yu']
xd, yd = s['xd'], s['yd']

In [18]:
file_hmi = files_hmi[0]
file_fdt = files_fdt[0]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

data_fdt = undistort(data_fdt, header_fdt, xd, yd)

view_fdt = View.from_header(header_fdt)
view_fdt.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view_fdt.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

view_hmi = View.from_header(header_hmi)

transform = view_hmi.to_carrington(correct_mu=False) - view_fdt.to_carrington(correct_mu=False)
grid, alpha = transform(view_hmi.grid())

In [7]:
xi, yi = np.round(grid[0]).astype(int).clip(0,view_fdt.nx-1), np.round(grid[1]).astype(int).clip(0,view_fdt.ny-1)

Q = np.zeros_like(data_fdt)
N = np.zeros_like(data_fdt)

for i in range(0,xi.shape[0]):
    for j in range(0,xi.shape[1]):
        Q[xi[i,j], yi[i,j]] += data_hmi[i,j]
        N[xi[i,j], yi[i,j]] += 1

Q /= N

/tmp/ipykernel_37484/3049325255.py:1: RuntimeWarning: invalid value encountered in cast
  xi, yi = np.round(grid[0]).astype(int).clip(0,view_fdt.nx-1), np.round(grid[1]).astype(int).clip(0,view_fdt.ny-1)
/tmp/ipykernel_37484/3049325255.py:11: RuntimeWarning: invalid value encountered in divide
  Q /= N


In [8]:
plt.figure(figsize=(10,10))
plt.imshow(N, vmin=0, vmax=100)

In [9]:
plt.figure(figsize=(10,10))
plt.imshow(Q, 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()

In [19]:
X = gaussian_filter(Q, sigma=0.7)
Y = data_fdt

xx, yy = [0], [0]

for q in np.arange(2., 30.25, 0.5):
    w = np.sqrt(X ** 2 + Y ** 2)
    t = np.where(w < q ** 2)
    x, y = X[t], Y[t]
    w = 1#w[t]

    A = np.array([[np.mean(w * x ** 2), np.mean(w * x * y)],
                  [np.mean(w * x * y), np.mean(w * y ** 2)]])

    vals, vecs = np.linalg.eigh(A)
    u, v = vecs[:, 1]
    u, v = u * q ** 2 * np.sign(u), v * q ** 2 * np.sign(u)

    xx = [-u] + xx + [u]
    yy = [-v] + yy + [v]

print(u / v)


xx = np.array(xx)
yy = np.array(yy)

1.1402145510801769


In [20]:
plt.figure(figsize=(10,10))
plt.plot([-400,400], [-400,400], color='black', lw=0.5)
plt.scatter(X, Y, s=0.05)
plt.plot([-u,u],[-v,v], '--', color='black', lw=1.5)
#plt.plot(xx, yy, '--', color='black', lw=1)
plt.plot()

plt.xlabel('HMI, G')
plt.ylabel('FDT, G')

#plt.xlim(-40,40)
#plt.ylim(-40,40)

plt.xlim(-200,200)
plt.ylim(-200,200)



plt.grid(True)
plt.tight_layout()

In [21]:
plt.figure(figsize=(10,10))
plt.imshow(X[300:500, 200:400], 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()

In [22]:
plt.figure(figsize=(10,10))
plt.imshow(Y[300:500, 200:400], 'hmimag', vmin=-1000, vmax=1000)
plt.axis('off')
plt.tight_layout()